In [120]:
import pandas as pd
import warnings
warnings.simplefilter(action='ignore',category=UserWarning)
anime_data = pd.read_csv('anime[1].csv')

In [121]:
anime_data.isnull().sum()

anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

In [122]:
# Filling missing numeric values with mean
anime_data['rating'].fillna(anime_data['rating'].mean(), inplace=True)
# Dropping rows with missing genre data
anime_data.dropna(subset=['genre'], inplace=True)

In [123]:
anime_data.info()
anime_data.describe()

<class 'pandas.core.frame.DataFrame'>
Index: 12232 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12232 non-null  int64  
 1   name      12232 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12210 non-null  object 
 4   episodes  12232 non-null  object 
 5   rating    12232 non-null  float64
 6   members   12232 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 764.5+ KB


,anime_id,rating,members
count,12232.000000,12232.000000,1.223200e+04
mean,13970.041530,6.478187,1.815922e+04
std,11415.656332,1.014818,5.494528e+04
min,1.000000,1.670000,5.000000e+00
25%,3465.750000,5.900000,2.297500e+02
50%,10199.500000,6.550000,1.572000e+03
75%,24544.500000,7.170000,9.494000e+03
max,34527.000000,10.000000,1.013917e+06


In [124]:
anime_data['genre'].unique()
anime_data['type'].unique()

array(['Movie', 'TV', 'OVA', 'Special', 'Music', 'ONA', nan], dtype=object)

In [125]:
anime_data['genre'].fillna('', inplace=True)

In [126]:
anime_data.dropna(subset=['genre'], inplace=True)

In [127]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer(tokenizer=lambda x: x.split(', '))
genre_matrix = vectorizer.fit_transform(anime_data['genre'])

In [128]:
anime_data = pd.get_dummies(anime_data, columns=['type'])

In [129]:
print(anime_data.columns)

Index(['anime_id', 'name', 'genre', 'episodes', 'rating', 'members',
       'type_Movie', 'type_Music', 'type_ONA', 'type_OVA', 'type_Special',
       'type_TV'],
      dtype='object')


In [130]:
print(anime_data[['rating', 'episodes']].head(10))
print(anime_data['rating'].unique())  # Check unique values in the rating column
print(anime_data['episodes'].unique())  # Check unique values in the episodes column

   rating episodes
0    9.37        1
1    9.26       64
2    9.25       51
3    9.17       24
4    9.16       51
5    9.15       10
6    9.13      148
7    9.11      110
8    9.10        1
9    9.11       13
[ 9.37        9.26        9.25        9.17        9.16        9.15
  9.13        9.11        9.1         9.06        9.05        9.04
  8.98        8.93        8.92        8.88        8.84        8.83
  8.82        8.81        8.8         8.78        8.77        8.76
  8.75        8.74        8.73        8.72        8.71        8.69
  8.68        8.67        8.66        8.65        8.64        8.62
  8.61        8.6         8.59        8.58        8.57        8.56
  8.55        8.54        8.53        8.52        8.51        8.5
  8.49        8.48        8.47        8.46        8.45        8.44
  8.43        8.42        8.41        8.4         8.39        8.38
  8.37        8.36        8.35        8.34        8.33        8.32
  8.31        8.3         8.29        8.28        8.27 

In [131]:
import numpy as np

# Convert 'Unknown' or any other invalid entries to NaN
anime_data['rating'] = pd.to_numeric(anime_data['rating'], errors='coerce')
anime_data['episodes'] = pd.to_numeric(anime_data['episodes'], errors='coerce')

In [132]:
anime_data.dropna(subset=['rating', 'episodes'], inplace=True)

In [133]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
anime_data[['rating', 'episodes']] = scaler.fit_transform(anime_data[['rating', 'episodes']])

In [134]:
print("Genre Matrix shape:", genre_matrix.shape)
print("Anime Data shape:", anime_data[['rating', 'episodes']].shape)

Genre Matrix shape: (12232, 43)
Anime Data shape: (11903, 2)


In [135]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import hstack

# Load your dataset here
# anime_data = pd.read_csv('your_anime_dataset.csv')

# Check for NaN values in the genre column
print(anime_data['genre'].isna().sum())  # Display the number of NaNs in the genre column

# Drop rows with NaN in the genre column
anime_data.dropna(subset=['genre'], inplace=True)

# Drop rows with NaN in rating or episodes as before
anime_data.dropna(subset=['rating', 'episodes'], inplace=True)

# Reset index to maintain alignment
anime_data.reset_index(drop=True, inplace=True)

# Create the genre matrix
vectorizer = CountVectorizer(tokenizer=lambda x: x.split(', '))
genre_matrix = vectorizer.fit_transform(anime_data['genre'])

# Normalize the numeric columns
scaler = MinMaxScaler()
anime_data[['rating', 'episodes']] = scaler.fit_transform(anime_data[['rating', 'episodes']])

# Combine the features
combined_features = hstack([genre_matrix, anime_data[['rating', 'episodes']].values])

# Confirm the shapes
print("Genre Matrix shape:", genre_matrix.shape)
print("Anime Data shape:", anime_data[['rating', 'episodes']].shape)

0
Genre Matrix shape: (11903, 43)
Anime Data shape: (11903, 2)


In [136]:
print(anime_data.isna().sum())  # Display the count of NaN values for all columns

anime_id        0
name            0
genre           0
episodes        0
rating          0
members         0
type_Movie      0
type_Music      0
type_ONA        0
type_OVA        0
type_Special    0
type_TV         0
dtype: int64


In [137]:
from scipy.sparse import hstack
combined_features = hstack([genre_matrix, anime_data[['rating', 'episodes']]])

In [138]:
from sklearn.metrics.pairwise import cosine_similarity
cosine_sim = cosine_similarity(combined_features, combined_features)

In [139]:
def get_recommendations(anime_title, cosine_sim=cosine_sim):
    idx = anime_data.index[anime_data['name'] == anime_title][0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:11]  # Top 10 recommendations
    anime_indices = [i[0] for i in sim_scores]
    return anime_data['name'].iloc[anime_indices]

In [140]:
anime_title = "Naruto"
recommendations = get_recommendations(anime_title)

# Print the top 10 recommendations
print(f"Top recommendations for '{anime_title}':")
print(recommendations)

Top recommendations for 'Naruto':
1098    Boruto: Naruto the Movie - Naruto ga Hokage ni...
484                              Boruto: Naruto the Movie
1336                                          Naruto x UT
1465          Naruto: Shippuuden Movie 4 - The Lost Tower
1566    Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...
2445                 Naruto Shippuuden: Sunny Side Battle
2977    Naruto Soyokazeden Movie: Naruto to Mashin to ...
174                                Katekyo Hitman Reborn!
7572                              Kyutai Panic Adventure!
7773                        Battle Spirits: Ryuuko no Ken
Name: name, dtype: object


In [141]:
def get_filtered_recommendations(anime_title, threshold=0.7):
    idx = anime_data.index[anime_data['name'] == anime_title][0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = [s for s in sim_scores if s[1] > threshold]
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[:10]  # Top 10 recommendations
    anime_indices = [i[0] for i in sim_scores]
    return anime_data['name'].iloc[anime_indices]

In [142]:
anime_title = "Naruto"  
recommendations = get_filtered_recommendations(anime_title, threshold=0.7)
print(f"Top recommendations for '{anime_title}':")
print(recommendations)

Top recommendations for 'Naruto':
838                                                Naruto
1098    Boruto: Naruto the Movie - Naruto ga Hokage ni...
484                              Boruto: Naruto the Movie
1336                                          Naruto x UT
1465          Naruto: Shippuuden Movie 4 - The Lost Tower
1566    Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...
2445                 Naruto Shippuuden: Sunny Side Battle
2977    Naruto Soyokazeden Movie: Naruto to Mashin to ...
174                                Katekyo Hitman Reborn!
7572                              Kyutai Panic Adventure!
Name: name, dtype: object


In [143]:
from sklearn.model_selection import train_test_split
train_data, test_data = train_test_split(anime_data, test_size=0.2, random_state=42)

In [144]:
def precision_at_k(recommended_list, relevant_list, k):
    recommended_k = recommended_list[:k]
    relevant_and_recommended = [r for r in recommended_k if r in relevant_list]
    return len(relevant_and_recommended) / k

In [145]:
recommendations = get_filtered_recommendations("Kyutai Panic Adventure!")
print(recommendations)

7572                              Kyutai Panic Adventure!
6923                                         Variable Geo
4552                    Street Fighter Zero The Animation
2977    Naruto Soyokazeden Movie: Naruto to Mashin to ...
2445                 Naruto Shippuuden: Sunny Side Battle
2403    Naruto: Honoo no Chuunin Shiken! Naruto vs. Ko...
1789                                       Rekka no Honoo
1566    Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...
1465          Naruto: Shippuuden Movie 4 - The Lost Tower
1336                                          Naruto x UT
Name: name, dtype: object


In [146]:
def evaluate_recommendations(anime_title):
    recommendations = get_filtered_recommendations(anime_title)
    recommended_ratings = anime_data[anime_data['name'].isin(recommendations)]['rating']
    avg_rating = recommended_ratings.mean()
    return avg_rating

average_rating = evaluate_recommendations("Kyutai Panic Adventure!")
print(f"Average rating of recommended anime: {average_rating}")

Average rating of recommended anime: 0.6327731092436976


In [147]:
def evaluate_episode_similarity(anime_title):
    recommendations = get_filtered_recommendations(anime_title)
    recommended_episodes = anime_data[anime_data['name'].isin(recommendations)]['episodes']
    avg_episodes = recommended_episodes.mean()
    return avg_episodes

average_episodes = evaluate_episode_similarity("Kyutai Panic Adventure!")
print(f"Average episode count of recommended anime: {average_episodes}")

Average episode count of recommended anime: 0.0023665382498624108


In [148]:
def precision_at_k(recommendations, k=10, threshold=8.0):
    liked_anime = anime_data[anime_data['rating'] > threshold]['name']
    relevant_recommendations = [anime for anime in recommendations if anime in liked_anime]
    precision = len(relevant_recommendations) / k
    return precision

recommendations = get_filtered_recommendations("Kyutai Panic Adventure!")
precision = precision_at_k(recommendations)
print(f"Precision at 10: {precision}")

Precision at 10: 0.0


In [ ]:
#Interview Questions
#Difference Between User-Based and Item-Based Collaborative Filtering:
#User-Based Collaborative Filtering: Recommends items to a user based on the preferences of similar users. The idea is that users with similar tastes tend to like the same items. It computes the similarity between users and suggests items that similar users liked.
#Item-Based Collaborative Filtering: Recommends items based on the similarity between items themselves. It looks at items a user has liked and finds similar items based on user interactions with these items. This approach is often more stable because user preferences change, but item properties usually don’t.

In [ ]:
#What is Collaborative Filtering, and How Does it Work?:
#Collaborative Filtering is a recommendation technique that makes predictions about a user's interests by collecting preferences or taste information from many users. It assumes that if user A has a similar interest to user B, then user A is likely to like things that user B likes.
#It works by either:
#Finding similar users (user-based) and recommending what those users liked.
#Finding similar items (item-based) and recommending similar items based on past user interactions